1. Crear el DataFrame con los datos del caso

In [10]:
import pandas as pd
import numpy as np

# Paso 1: definir los datos según el enunciado
data = {
    'ID': [1, 2, 3, 4],
    'Edad': [25, 45, 30, 40],
    'Ciudad': ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Ingresos': [30000, 50000, np.nan, 40000]
}

# Paso 2: crear el DataFrame
df = pd.DataFrame(data)

# Paso 3: ver los datos originales
print("Datos originales:\n", df)


Datos originales:
    ID  Edad     Ciudad  Ingresos
0   1    25     Madrid   30000.0
1   2    45    Sevilla   50000.0
2   3    30     Madrid       NaN
3   4    40  Barcelona   40000.0


2. Imputar el valor faltante en Ingresos con la media

In [11]:
from sklearn.impute import SimpleImputer

# Paso 4: crear el imputador con estrategia 'mean'
imputer = SimpleImputer(strategy='mean')

# Paso 5: ajustar y transformar la columna 'Ingresos'
df['Ingresos'] = imputer.fit_transform(df[['Ingresos']])

# Paso 6: ver el resultado
print("\nDatos con ingresos imputados:\n", df)



Datos con ingresos imputados:
    ID  Edad     Ciudad  Ingresos
0   1    25     Madrid   30000.0
1   2    45    Sevilla   50000.0
2   3    30     Madrid   40000.0
3   4    40  Barcelona   40000.0


Interpretación: la media de ingresos se calcula con las filas que tienen dato (30000, 50000, 40000), y ese valor reemplaza el NaN del registro 3.

3. Codificar Ciudad con Label Encoding y One-Hot

In [12]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Paso 7: Label Encoding
le = LabelEncoder()
df['Ciudad_Label'] = le.fit_transform(df['Ciudad'])

print("\nLabel Encoding:\n", df[['Ciudad', 'Ciudad_Label']])



Label Encoding:
       Ciudad  Ciudad_Label
0     Madrid             1
1    Sevilla             2
2     Madrid             1
3  Barcelona             0


Interpretación: cada ciudad se reemplaza por un entero (0, 1, 2, …); útil para algunos modelos, pero introduce un orden que no es real entre categorías.

In [13]:
# Paso 8: One-Hot Encoding (para sklearn reciente)
ohe = OneHotEncoder(sparse_output=False)

# Paso 9: ajustar y transformar
ciudad_encoded = ohe.fit_transform(df[['Ciudad']])

# Paso 10: convertir a DataFrame con nombres de columnas
df_ohe = pd.DataFrame(
    ciudad_encoded,
    columns=ohe.get_feature_names_out(['Ciudad'])
)

print("\nOne-Hot Encoding:\n", df_ohe)



One-Hot Encoding:
    Ciudad_Barcelona  Ciudad_Madrid  Ciudad_Sevilla
0               0.0            1.0             0.0
1               0.0            0.0             1.0
2               0.0            1.0             0.0
3               1.0            0.0             0.0


Interpretación: se crean columnas binarias del tipo Ciudad_Barcelona, Ciudad_Madrid, Ciudad_Sevilla con valores 0/1, sin imponer orden entre las categorías.

4. Variables Dummy para Ciudad

In [14]:
# Paso 11: crear variables dummy (drop_first para evitar colinealidad)
df_dummy = pd.get_dummies(df['Ciudad'], drop_first=True)

print("\nVariables Dummy:\n", df_dummy)



Variables Dummy:
    Madrid  Sevilla
0    True    False
1   False     True
2    True    False
3   False    False


Interpretación: se generan columnas para todas las ciudades menos una; cuando todas las columnas dummies valen 0, corresponde a la categoría que se dejó como referencia.

5. Escalado Min-Max y Z-Score de Edad e Ingresos


In [15]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Paso 12: crear escalador Min-Max
scaler_minmax = MinMaxScaler()

# Paso 13: ajustar y transformar Edad e Ingresos
df[['Edad_MinMax', 'Ingresos_MinMax']] = scaler_minmax.fit_transform(
    df[['Edad', 'Ingresos']]
)

print("\nMin-Max Scaling:\n", df[['Edad_MinMax', 'Ingresos_MinMax']])



Min-Max Scaling:
    Edad_MinMax  Ingresos_MinMax
0         0.00              0.0
1         1.00              1.0
2         0.25              0.5
3         0.75              0.5


In [16]:

# Paso 14: crear escalador estándar (Z-Score)
scaler_std = StandardScaler()

# Paso 15: ajustar y transformar
df[['Edad_Std', 'Ingresos_Std']] = scaler_std.fit_transform(
    df[['Edad', 'Ingresos']]
)

print("\nEstandarización:\n", df[['Edad_Std', 'Ingresos_Std']])



Estandarización:
    Edad_Std  Ingresos_Std
0 -1.264911     -1.414214
1  1.264911      1.414214
2 -0.632456      0.000000
3  0.632456      0.000000


6. Respuestas breves de reflexión

- ¿Por qué es importante preprocesar y escalar antes de entrenar un modelo?

    - Porque muchos algoritmos no aceptan valores nulos, necesitan solo números y son sensibles a las escalas.

    - Sin imputar, codificar y escalar, el modelo puede no entrenar, entrenar con errores o dar más peso a variables solo porque tienen valores grandes (por ejemplo, ingresos vs. edad).
​
- Diferencias entre normalización Min-Max y estandarización Z-Score

    - La normalización Min-Max lleva los datos a un rango acotado (generalmente ), conservando la forma de la distribución pero “aplastando” al rango.
​
    - La estandarización centra los datos en 0 y los escala para que la desviación estándar sea 1; no fija un rango concreto pero hace que la mayoría de valores queden alrededor de −3 y 3.
​

7. Código completo para .py o .ipynb

In [20]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler

# 1. Datos
data = {
    'ID': [1, 2, 3, 4],
    'Edad': [25, 45, 30, 40],
    'Ciudad': ['Madrid', 'Sevilla', 'Madrid', 'Barcelona'],
    'Ingresos': [30000, 50000, np.nan, 40000]
}

df = pd.DataFrame(data)
print("Datos originales:\n", df)

# 2. Imputación de Ingresos con la media
imputer = SimpleImputer(strategy='mean')
df['Ingresos'] = imputer.fit_transform(df[['Ingresos']])
print("\nDatos con ingresos imputados:\n", df)

# 3. Label Encoding de Ciudad
le = LabelEncoder()
df['Ciudad_Label'] = le.fit_transform(df['Ciudad'])
print("\nLabel Encoding:\n", df[['Ciudad', 'Ciudad_Label']])

# 4. One-Hot Encoding de Ciudad
ohe = OneHotEncoder(sparse_output=False)
ciudad_encoded = ohe.fit_transform(df[['Ciudad']])
df_ohe = pd.DataFrame(ciudad_encoded, columns=ohe.get_feature_names_out(['Ciudad']))
print("\nOne-Hot Encoding:\n", df_ohe)

# 5. Variables Dummy de Ciudad
df_dummy = pd.get_dummies(df['Ciudad'], drop_first=True)
print("\nVariables Dummy:\n", df_dummy)

# 6. Escalado Min-Max
scaler_minmax = MinMaxScaler()
df[['Edad_MinMax', 'Ingresos_MinMax']] = scaler_minmax.fit_transform(df[['Edad', 'Ingresos']])
print("\nMin-Max Scaling:\n", df[['Edad_MinMax', 'Ingresos_MinMax']])

# 7. Estandarización Z-Score
scaler_std = StandardScaler()
df[['Edad_Std', 'Ingresos_Std']] = scaler_std.fit_transform(df[['Edad', 'Ingresos']])
print("\nEstandarización:\n", df[['Edad_Std', 'Ingresos_Std']])

# 8. Guardar el DataFrame final a CSV
df.to_csv('clientes_preprocesados.csv', index=False)
print("\nArchivo 'clientes_preprocesados.csv' guardado en la carpeta de trabajo.")

Datos originales:
    ID  Edad     Ciudad  Ingresos
0   1    25     Madrid   30000.0
1   2    45    Sevilla   50000.0
2   3    30     Madrid       NaN
3   4    40  Barcelona   40000.0

Datos con ingresos imputados:
    ID  Edad     Ciudad  Ingresos
0   1    25     Madrid   30000.0
1   2    45    Sevilla   50000.0
2   3    30     Madrid   40000.0
3   4    40  Barcelona   40000.0

Label Encoding:
       Ciudad  Ciudad_Label
0     Madrid             1
1    Sevilla             2
2     Madrid             1
3  Barcelona             0

One-Hot Encoding:
    Ciudad_Barcelona  Ciudad_Madrid  Ciudad_Sevilla
0               0.0            1.0             0.0
1               0.0            0.0             1.0
2               0.0            1.0             0.0
3               1.0            0.0             0.0

Variables Dummy:
    Madrid  Sevilla
0    True    False
1   False     True
2    True    False
3   False    False

Min-Max Scaling:
    Edad_MinMax  Ingresos_MinMax
0         0.00            